# Colab: Text Segments for Binary Classification

This notebook is designed for Google Colab.

It extracts these segments from the structured `text` column:
- `title`: content between `[TITLE]` and `[HASHTAG]`
- `keywords`: content between `[KEYWORDS]` and `[TAGS]`
- `tags`: content after `[TAGS]`

By default, the model uses `title + keywords + tags` as the text input.

In [ ]:
import sys
assert 'google.colab' in sys.modules, 'Please run this notebook in Google Colab.'
print('Running in Colab')

In [ ]:
%cd /content/
!rm -rf project_viral
!git clone https://github.com/Zmjjkk880/project_viral
%cd /content/project_viral
!ls

In [ ]:
!pip -q install sentence-transformers scikit-learn pandas numpy torch

In [ ]:
# If you do not already have the processed CSV, run this cell.
!python preprocess.py \
  --views-threshold 20000 \
  --engagement-threshold 0.06

In [ ]:
import os
import random
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

os.environ.setdefault('HF_HOME', '/content/.hf_cache')

TARGET_COLUMN = 'viral'
TEXT_COLUMN = 'text'
RANDOM_STATE = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def extract_segment(series: pd.Series, start_tag: str, end_tag: str | None = None) -> pd.Series:
    pattern = re.escape(start_tag) + r'(.*?)'
    if end_tag is not None:
        pattern += re.escape(end_tag)
    extracted = series.fillna('').astype(str).str.extract(pattern, expand=False).fillna('')
    return extracted.str.strip()


def build_segment_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    base_text = df[TEXT_COLUMN].fillna('').astype(str)
    out_df = df.copy()
    out_df['title_text'] = extract_segment(base_text, '[TITLE]', '[HASHTAG]')
    out_df['hashtag_text'] = extract_segment(base_text, '[HASHTAG]', '[KEYWORDS]')
    out_df['keywords_text'] = extract_segment(base_text, '[KEYWORDS]', '[TAGS]')
    out_df['tags_text'] = extract_segment(base_text, '[TAGS]')
    out_df['full_text'] = base_text
    return out_df


def compose_model_text(df: pd.DataFrame, segment_mode: str) -> pd.Series:
    if segment_mode == 'title':
        parts = [df['title_text']]
    elif segment_mode == 'keywords':
        parts = [df['keywords_text']]
    elif segment_mode == 'tags':
        parts = [df['tags_text']]
    elif segment_mode == 'hashtag':
        parts = [df['hashtag_text']]
    elif segment_mode == 'title_keywords':
        parts = [df['title_text'], df['keywords_text']]
    elif segment_mode == 'title_tags':
        parts = [df['title_text'], df['tags_text']]
    elif segment_mode == 'keywords_tags':
        parts = [df['keywords_text'], df['tags_text']]
    elif segment_mode == 'title_keywords_tags':
        parts = [df['title_text'], df['keywords_text'], df['tags_text']]
    elif segment_mode == 'full_text':
        parts = [df['full_text']]
    else:
        raise ValueError(f'Unsupported segment_mode: {segment_mode}')

    combined = parts[0].copy()
    for part in parts[1:]:
        combined = combined.str.cat(part, sep=' ')
    return combined.str.replace(r'\s+', ' ', regex=True).str.strip()


class TextOnlyClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float) -> None:
        super().__init__()
        hidden_dim_2 = max(hidden_dim // 2, 1)
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim_2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim_2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x).squeeze(1)


def encode_text(train_texts, test_texts, model_name: str, encode_batch_size: int, normalize_embeddings: bool):
    encoder = SentenceTransformer(model_name, device=str(DEVICE))
    train_embeddings = encoder.encode(
        train_texts,
        batch_size=encode_batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize_embeddings,
    ).astype(np.float32)
    test_embeddings = encoder.encode(
        test_texts,
        batch_size=encode_batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize_embeddings,
    ).astype(np.float32)
    return train_embeddings, test_embeddings


def create_dataloader(features: np.ndarray, labels: np.ndarray, shuffle: bool, batch_size: int):
    dataset = TensorDataset(
        torch.tensor(features, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def evaluate(model: nn.Module, dataloader: DataLoader, criterion: nn.Module):
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            probs = torch.sigmoid(logits)
            total_loss += loss.item() * batch_x.size(0)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)
    return avg_loss, np.array(all_probs), np.array(all_labels)


def find_best_threshold(labels: np.ndarray, probs: np.ndarray):
    best_threshold = 0.5
    best_macro_f1 = float('-inf')
    for threshold in np.arange(0.05, 0.96, 0.01):
        preds = (probs >= threshold).astype(int)
        macro_f1 = f1_score(labels, preds, average='macro')
        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_threshold = float(threshold)
    return best_threshold, best_macro_f1


def train_one_run(
    data_path,
    segment_mode='title_keywords_tags',
    bert_model_name='sentence-transformers/all-MiniLM-L6-v2',
    normalize_embeddings=False,
    batch_size=64,
    epochs=10,
    learning_rate=1e-3,
    hidden_dim=128,
    dropout=0.3,
    test_size=0.2,
    threshold=0.5,
    text_encode_batch_size=64,
):
    set_seed(RANDOM_STATE)
    df = pd.read_csv(data_path)
    print(f'Dataset shape: {df.shape}')
    print(f'Positive rate: {df[TARGET_COLUMN].mean():.4f}')

    df = build_segment_dataframe(df)
    df['model_text'] = compose_model_text(df, segment_mode)
    print(f'Segment mode: {segment_mode}')
    print('Example model_text:')
    print(df['model_text'].iloc[0][:300])

    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=df[TARGET_COLUMN],
    )

    train_embeddings, test_embeddings = encode_text(
        train_df['model_text'].tolist(),
        test_df['model_text'].tolist(),
        model_name=bert_model_name,
        encode_batch_size=text_encode_batch_size,
        normalize_embeddings=normalize_embeddings,
    )

    y_train = train_df[TARGET_COLUMN].to_numpy(dtype=np.float32)
    y_test = test_df[TARGET_COLUMN].to_numpy(dtype=np.float32)

    train_loader = create_dataloader(train_embeddings, y_train, shuffle=True, batch_size=batch_size)
    test_loader = create_dataloader(test_embeddings, y_test, shuffle=False, batch_size=batch_size)

    model = TextOnlyClassifier(
        input_dim=train_embeddings.shape[1],
        hidden_dim=hidden_dim,
        dropout=dropout,
    ).to(DEVICE)

    positive_count = y_train.sum()
    negative_count = len(y_train) - positive_count
    pos_weight = torch.tensor([negative_count / max(positive_count, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_auc = float('-inf')
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * batch_x.size(0)

        avg_train_loss = total_train_loss / len(train_loader.dataset)
        test_loss, test_probs_epoch, test_labels_epoch = evaluate(model, test_loader, criterion)
        test_auc = roc_auc_score(test_labels_epoch, test_probs_epoch)

        if test_auc > best_auc:
            best_auc = test_auc
            best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f'Epoch {epoch + 1}/{epochs} | '
            f'Train Loss: {avg_train_loss:.4f} | '
            f'Test Loss: {test_loss:.4f} | '
            f'Test ROC-AUC: {test_auc:.4f}'
        )

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_probs, test_labels = evaluate(model, test_loader, criterion)
    fixed_preds = (test_probs >= threshold).astype(int)
    tuned_threshold, tuned_macro_f1 = find_best_threshold(test_labels, test_probs)
    tuned_preds = (test_probs >= tuned_threshold).astype(int)

    result = {
        'segment_mode': segment_mode,
        'test_loss': float(test_loss),
        'test_auc': float(roc_auc_score(test_labels, test_probs)),
        'fixed_accuracy': float(accuracy_score(test_labels, fixed_preds)),
        'fixed_macro_f1': float(f1_score(test_labels, fixed_preds, average='macro')),
        'best_threshold': float(tuned_threshold),
        'best_macro_f1': float(tuned_macro_f1),
    }

    print('\nFinal Result:')
    for k, v in result.items():
        print(f'{k}: {v}')

    print('\nClassification Report @ tuned threshold:\n')
    print(classification_report(test_labels, tuned_preds, digits=4))
    return result

In [ ]:
# Single run config
DATA_PATH = '/content/project_viral/data/processed/processed_data_20000_0.06.csv'
SEGMENT_MODE = 'title_keywords_tags'  # options: title, keywords, tags, hashtag, title_keywords, title_tags, keywords_tags, title_keywords_tags, full_text
BERT_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
NORMALIZE_EMBEDDINGS = False
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-3
HIDDEN_DIM = 128
DROPOUT = 0.3
THRESHOLD = 0.5

In [ ]:
result = train_one_run(
    data_path=DATA_PATH,
    segment_mode=SEGMENT_MODE,
    bert_model_name=BERT_MODEL_NAME,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    threshold=THRESHOLD,
)

In [ ]:
# Optional: compare multiple segment combinations in one run
segment_modes = [
    'title',
    'keywords',
    'tags',
    'title_keywords',
    'keywords_tags',
    'title_keywords_tags',
    'full_text',
]

all_results = []
for mode in segment_modes:
    print('\n' + '=' * 80)
    print('Running mode:', mode)
    res = train_one_run(
        data_path=DATA_PATH,
        segment_mode=mode,
        bert_model_name=BERT_MODEL_NAME,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        threshold=THRESHOLD,
    )
    all_results.append(res)

results_df = pd.DataFrame(all_results).sort_values(['test_auc', 'best_macro_f1'], ascending=False)
results_df